In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import os
from sklearn.metrics import classification_report
from pathlib import Path

DATA_PATH = Path("../data")

In [ ]:

# ===============================
# Chargement des fichiers CSV
# ===============================

train_df = pd.read_csv(DATA_PATH / "train_features_cleaned.csv", index_col=0)
test_df = pd.read_csv(DATA_PATH / "test_features_cleaned.csv", index_col=0)
target_df = pd.read_csv(DATA_PATH / "train_target_corr.csv", index_col=0)


# 1. Préparation des matrices 
X = train_df.astype(float).values
y = target_df.values.ravel()
X_test = test_df.astype(float).values

# 2. Configuration de la validation croisée
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros((len(X_test), len(np.unique(y))))

# Paramètres optimisés pour LightGBM Multiclasse
lgb_params = {
    'objective': 'multiclass',
    'class_weight': 'balanced',
    'num_class': len(np.unique(y)),
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,

    # Valeurs optimisées par Optuna
    'learning_rate': 0.051,
    'num_leaves': 253,
    'max_depth': 11,
    'min_data_in_leaf': 50,
    'lambda_l1': 4.369779325101578e-08,
    'lambda_l2': 1.9956616320033012,
    'feature_fraction': 0.6573768736676399,
    'bagging_fraction': 0.8811983240188239,
    'bagging_freq': 3
}

print(f"Début de l'entraînement LightGBM sur {N_SPLITS} Folds...")

# 3. Boucle d'entraînement
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold + 1}/{N_SPLITS} ---")

    X_train_f, y_train_f = X[train_idx], y[train_idx]
    X_val_f, y_val_f = X[val_idx], y[val_idx]

    dtrain = lgb.Dataset(X_train_f, label=y_train_f)
    dval = lgb.Dataset(X_val_f, label=y_val_f, reference=dtrain)

    # Entraînement avec Early Stopping
    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=2000,
        valid_sets=[dtrain, dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    # Prédictions sur la validation (pour calculer ton score local)
    val_preds_proba = model.predict(X_val_f)
    oof_preds[val_idx] = np.argmax(val_preds_proba, axis=1)

    # Prédictions sur le test set (on accumule les probabilités)
    test_preds += model.predict(X_test) / N_SPLITS

# 4. Évaluation globale Out-Of-Fold (OOF)
accuracy = accuracy_score(y, oof_preds)
f1_macro = f1_score(y, oof_preds, average='macro')

print("RÉSULTATS FINAUX (Out-Of-Fold)")
print(f"Accuracy Globale : {accuracy:.5f}")
print(f"F1-Score (Macro) : {f1_macro:.5f}")

# 5. Création du fichier de soumission
# On prend la classe avec la plus haute probabilité moyenne
final_test_predictions_encoded = np.argmax(test_preds, axis=1)

# Création du DataFrame de soumission
submission = pd.DataFrame({
    'Id': test_df.index,
    'change_type': final_test_predictions_encoded
})

submission.to_csv(DATA_PATH / "submission_lgb_final.csv", index=False)
print("Fichier 'submission_lgb_final.csv' généré avec succès !")


#le diagnostic
report = classification_report(y, oof_preds, digits=5)
print(report)  